In [0]:
import json
from pathlib import Path
import pandas as pd

In [0]:
dbutils.widgets.text("raw_root", "/Volumes/proyecto_final_prueba/raw/weather","raw root")
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
spark.sql("USE CATALOG 'proyecto_final_prueba'")

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = 'bronze'
table = 'weather'

In [0]:
spark.sql(f"DROP DATABASE IF EXISTS {catalog}.{schema} cascade")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
  latitude DOUBLE,
  longitude DOUBLE,
  generationtime_ms DOUBLE,
  utc_offset_seconds INT,
  timezone STRING,
  timezone_abbreviation STRING,
  elevation DOUBLE,

  hourly_units STRUCT<
    time: STRING,
    temperature_2m: STRING,
    relative_humidity_2m: STRING,
    wind_speed_10m: STRING,
    weather_code: STRING
  >,

  hourly STRUCT<
    time: ARRAY<STRING>,
    temperature_2m: ARRAY<DOUBLE>,
    relative_humidity_2m: ARRAY<INT>,
    wind_speed_10m: ARRAY<DOUBLE>,
    weather_code: ARRAY<INT>
  >,

  ingestion_date TIMESTAMP,
  source_file STRING
)
""")

In [0]:
candidate_files = sorted(raw_root.glob("**/weather_*.json"))

records: list[dict] = []
for json_file in candidate_files:
    payload = json.loads(json_file.read_text(encoding="utf-8"))
    if isinstance(payload, list):
        records.extend(payload)
    else:
        records.append(payload)

df = pd.json_normalize(records)

df_spark = spark.createDataFrame(df)
df_spark.display()

In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")